# 🦙 Llama 3.1 8B — Varyant Test (FINAL)

Bu notebook **sadece Llama 3.1 8B Instruct'ı** varyant veri seti üzerinde test eder. Önceki tüm sorunlar (paket uyumsuzluğu, GPU OOM, JSON parse hataları, kesik skorlama) baştan çözüldü.

## 📋 Çalıştırma Sırası — ZORUNLU

Hücreleri **sırayla 1'den 13'e kadar** çalıştır. Atlama yok, sırayı bozma.

1. Paketleri kur (1 kez, runtime restart gerektirir)
2. RUNTIME RESTART (manuel)
3. Drive bağla + path'leri ayarla
4. GPU kontrolü + HF login
5. Veri seti yükle + rename
6. Embedding/NLI CPU'da yükle (GPU Llama'ya bırakılıyor)
7. Yardımcı fonksiyonlar (prompt + generation + parser)
8. Llama'yı yükle (4-bit)
9. Generation çalıştır (checkpoint'li, ~1.5 saat)
10. GPU'yu temizle
11. Checkpoint'li skorlama (her 10 soruda kayıt, kesilse devam eder)
12. Özet ve sonuç CSV
13. (Opsiyonel) Yeniden skorlama, sadece skorlama kesilirse

**Önkoşul:** `/content/drive/MyDrive/BitirmeProjesi/core/faithfulness_utils.py` ve varyant CSV dosyası Drive'da olmalı.

**GPU önerisi:** L4 (22 GB) > T4 (14 GB). A100 gereksiz, bakiye yer.

**Tahmini bakiye:** L4 ile ~8 unit, T4 ile ~3 unit.

---
## 📦 1. Paketleri kur (1 kez)

**Bu hücre çalıştıktan sonra mutlaka Runtime → Restart session yap, sonra 3. hücreden devam et.**
---

In [ ]:
# Sorunlu paketleri kaldır, uyumlu sürümleri kur
!pip uninstall -y transformers torch torchvision torchaudio -q
!pip install -q torch==2.5.1
!pip install -q transformers==4.46.0
!pip install -q accelerate==1.1.1 bitsandbytes==0.44.1 sentencepiece
!pip install -q sentence-transformers==3.3.1
!pip install -q tqdm scipy

print("\n" + "=" * 60)
print("✓ Paketler kuruldu.")
print("⚠️  ŞİMDİ ÖNEMLİ: Runtime → Restart session yap!")
print("   Restart'tan sonra 3. hücreden başla (bu hücreyi atla)")
print("=" * 60)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 140.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 113.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

---
## 🔄 2. RUNTIME RESTART (Manuel)

Üst menü → **Runtime → Restart session** → Yes

Restart'tan sonra **bu üst kısmı tekrar çalıştırma**, doğrudan **3. hücreye** geç.
---

---
## 🔌 3. Drive bağla + path'leri ayarla
---

In [ ]:
# Drive bağla ve faithfulness_utils'i import et
from google.colab import drive
drive.mount('/content/drive')

import sys, os, shutil, gc, json
from datetime import datetime

PROJECT_FOLDER = "/content/drive/MyDrive/BitirmeProjesi"
sys.path.insert(0, PROJECT_FOLDER)

# Eski cache'i temizle (yeni import garantisi)
pycache_path = f"{PROJECT_FOLDER}/core/__pycache__"
if os.path.exists(pycache_path):
    shutil.rmtree(pycache_path)
    print("✓ __pycache__ temizlendi")

utils_path = f"{PROJECT_FOLDER}/core/faithfulness_utils.py"
assert os.path.exists(utils_path), f"faithfulness_utils.py bulunamadı: {utils_path}"
print(f"✓ faithfulness_utils.py bulundu")

from core.faithfulness_utils import *
print("✓ faithfulness_utils fonksiyonları import edildi")

# Path'ler
CHECKPOINT_PATH = f"{PROJECT_FOLDER}/checkpoints/"
RESULTS_PATH = f"{PROJECT_FOLDER}/results/"
os.makedirs(CHECKPOINT_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

TODAY = datetime.now().strftime('%Y%m%d')
print(f"✓ Tarih: {TODAY}")
print(f"✓ Checkpoint dizini: {CHECKPOINT_PATH}")
print(f"✓ Results dizini: {RESULTS_PATH}")

Mounted at /content/drive
✓ __pycache__ temizlendi
✓ faithfulness_utils.py bulundu
✓ faithfulness_utils fonksiyonları import edildi
✓ Tarih: 20260517
✓ Checkpoint dizini: /content/drive/MyDrive/BitirmeProjesi/checkpoints/
✓ Results dizini: /content/drive/MyDrive/BitirmeProjesi/results/


---
## 🎮 4. GPU kontrolü + HF login
---

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from sentence_transformers import SentenceTransformer
from huggingface_hub import login
from google.colab import userdata

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name} ({vram:.1f} GB VRAM)")
else:
    raise RuntimeError("GPU yok! Runtime → Change runtime type → GPU seç (L4 önerilir)")

# HuggingFace login (Llama 3.1 için zorunlu, model erişimi onaylı olmalı)
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    login(token=HF_TOKEN)
    print("✓ HuggingFace login başarılı")
except Exception as e:
    print(f"⚠ HF token sorunu: {e}")
    print("  Colab Secrets → HF_TOKEN ekle, sonra tekrar çalıştır")

CUDA available: True
GPU: NVIDIA L4 (22.0 GB VRAM)
✓ HuggingFace login başarılı


---
## 📊 5. Varyant veri setini yükle + kolon rename
---

In [ ]:
import pandas as pd

# Varyant CSV'sini bul (önce tahmin edilen yol, sonra fallback arama)
DATA_PATH = f"{PROJECT_FOLDER}/Sorular e-tablo - Variants (1).csv"

if not os.path.exists(DATA_PATH):
    print(f"⚠ Tahmin edilen yol yok: {DATA_PATH}")
    print("  Otomatik arama...")
    candidates = [
        f for f in os.listdir(PROJECT_FOLDER)
        if f.endswith('.csv') and 'varyant' in f.lower()
    ]
    if candidates:
        DATA_PATH = f"{PROJECT_FOLDER}/{candidates[0]}"
        print(f"  ✓ Bulundu: {DATA_PATH}")
    else:
        print("\nKlasördeki CSV'ler:")
        for f in os.listdir(PROJECT_FOLDER):
            if f.endswith('.csv'):
                print(f"   - {f}")
        raise FileNotFoundError("Varyant CSV bulunamadı. DATA_PATH'i elle ayarla.")

# Config'i utils'e set et
set_config(
    vector_w=0.4,
    nli_w=0.6,
    llm_model="local-open-source",
    temp=0,
    max_tok=2048,
    checkpoint_path=CHECKPOINT_PATH
)

# Veriyi yükle ve preprocess et
df = load_and_preprocess_data(DATA_PATH)

# Varyant CSV'sinde 'variant_id' var, biz 'question_id' bekliyoruz → rename
if 'variant_id' in df.columns and 'question_id' not in df.columns:
    df = df.rename(columns={'variant_id': 'question_id'})
    print("✓ variant_id → question_id rename yapıldı")

DATASET_TAG = "varyant"

print(f"\n✓ Veri seti yüklendi: {len(df)} satır")
print(f"  Dosya: {DATA_PATH.split('/')[-1]}")
print(f"  İlk question_id: {df.iloc[0]['question_id']}")
print(f"  Kolonlar: {df.columns.tolist()}")

✓ Veri yüklendi: 257 satır
  Evidence parça sayıları: min=1, max=6, ortalama=3.0
✓ variant_id → question_id rename yapıldı

✓ Veri seti yüklendi: 257 satır
  Dosya: Sorular e-tablo - Variants (1).csv
  İlk question_id: TR-01-017-v1
  Kolonlar: ['question_id', 'base_question_id', 'category', 'difficulty', 'variant_type', 'morphological_feature', 'context', 'question', 'gold_evidence', 'gold_answer', 'gold_evidence_list', 'evidence_count']


---
## 🧠 6. Embedding ve NLI modellerini CPU'da yükle

GPU'yu Llama 3.1'e bırakıyoruz. Embedding/NLI sadece skorlamada kullanılacak ve CPU'da yeterince hızlı.
---

In [ ]:
EMBEDDING_MODEL = "intfloat/multilingual-e5-large"
NLI_MODEL = "joeddav/xlm-roberta-large-xnli"

print("► Embedding model CPU'da yükleniyor (~1 dk)...")
emb_model = SentenceTransformer(EMBEDDING_MODEL, device='cpu')
print(f"  ✓ {EMBEDDING_MODEL}")

print("► NLI pipeline CPU'da yükleniyor (~1 dk)...")
nli_pipe = pipeline(
    "zero-shot-classification",
    model=NLI_MODEL,
    device=-1   # CPU
)
print(f"  ✓ {NLI_MODEL}")

set_models(emb_model, nli_pipe, None, None, None)

# GPU bellek durumunu kontrol et (Llama için ne kadar yer var)
free = torch.cuda.mem_get_info()[0] / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"\n✓ Skorlama altyapısı hazır (CPU)")
print(f"  GPU boş: {free:.2f} / {total:.2f} GB (Llama'ya ayrıldı)")
if free < 10:
    print("  ⚠ DİKKAT: 10 GB altı GPU. Llama OOM yapabilir.")

► Embedding model CPU'da yükleniyor (~1 dk)...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

  ✓ joeddav/xlm-roberta-large-xnli

✓ Skorlama altyapısı hazır (CPU)
  GPU boş: 21.84 / 22.03 GB (Llama'ya ayrıldı)


---
## 🛠️ 7. Yardımcı fonksiyonlar (prompt, generation, parser)
---

In [ ]:
# ---------- 7.1 — İyileştirilmiş prompt (JSON-only kuralı) ----------
OPEN_SOURCE_SYSTEM_PROMPT = SYSTEM_PROMPT + """

ÇOK ÖNEMLİ FORMAT KURALI:
- Cevabını YALNIZCA tek bir JSON nesnesi olarak ver.
- JSON'u markdown kod bloğu (```json) içinde verme.
- JSON'dan önce veya sonra HİÇBİR açıklama, selamlama, özet ekleme.
- 'reasoning' alanını TEK bir string olarak yaz, ayrı string listesi olarak değil.
- 'reasoning' içinde satır geçişi kullanmak istiyorsan \\n yaz, literal newline değil.
- Cevabını JSON'un kapanış süslü parantezi ile bitir.
"""

def build_user_prompt(context: str, question: str) -> str:
    return f"""BAĞLAM:
{context}

SORU:
{question}

Yukarıdaki kurallara uygun, SADECE geçerli JSON formatında ve markdown sarmadan cevap ver."""


# ---------- 7.2 — Generation fonksiyonu (max_new_tokens=1024) ----------
def _supports_system_role(tokenizer) -> bool:
    try:
        tokenizer.apply_chat_template(
            [{"role": "system", "content": "test"}, {"role": "user", "content": "test"}],
            tokenize=False, add_generation_prompt=True,
        )
        return True
    except Exception:
        return False

def generate_response(model, tokenizer, context: str, question: str, max_new_tokens: int = 1024) -> str:
    user_prompt = build_user_prompt(context, question)
    if _supports_system_role(tokenizer):
        messages = [
            {"role": "system", "content": OPEN_SOURCE_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]
    else:
        combined = f"{OPEN_SOURCE_SYSTEM_PROMPT}\n\n{user_prompt}"
        messages = [{"role": "user", "content": combined}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


# ---------- 7.3 — 4 aşamalı fallback JSON parser ----------
import re as _re
import json as _json

_original_extract_json = extract_json_from_response

def _fb_markdown(raw):
    m = _re.search(r'```(?:json)?\s*(\{.*?\})\s*```', raw, _re.DOTALL)
    if m:
        try: return _json.loads(m.group(1))
        except _json.JSONDecodeError: return None
    return None

def _fb_literal_newline(raw):
    s = raw.strip()
    if not s.startswith('{'):
        idx = s.find('{')
        if idx == -1: return None
        s = s[idx:]
    depth, in_string, escape, end_idx = 0, False, False, -1
    for i, ch in enumerate(s):
        if escape: escape = False; continue
        if ch == '\\': escape = True; continue
        if ch == '"': in_string = not in_string; continue
        if in_string: continue
        if ch == '{': depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0: end_idx = i; break
    if end_idx == -1: return None
    candidate = s[:end_idx+1]
    result_chars = []
    in_string = False; escape = False
    for ch in candidate:
        if escape: result_chars.append(ch); escape = False; continue
        if ch == '\\': result_chars.append(ch); escape = True; continue
        if ch == '"': in_string = not in_string; result_chars.append(ch); continue
        if in_string and ch in '\n\r\t':
            result_chars.append({'\n':'\\n','\r':'\\r','\t':'\\t'}[ch])
        else:
            result_chars.append(ch)
    try: return _json.loads(''.join(result_chars))
    except _json.JSONDecodeError: return None

def _fb_concatenated(raw):
    s = raw.strip()
    if not s.startswith('{'):
        idx = s.find('{')
        if idx == -1: return None
        s = s[idx:]
    r_m = _re.search(r'"reasoning"\s*:\s*"([^"]*)"', s, _re.DOTALL)
    a_m = _re.search(r'"answer"\s*:\s*(?:"([^"]*)"|([-+]?\d+(?:\.\d+)?)|(true|false|null))', s)
    c_m = _re.search(r'"confidence"\s*:\s*(\d+(?:\.\d+)?)', s)
    if r_m is None: return None
    reasoning = r_m.group(1)
    eor = r_m.end()
    middle = s[eor:a_m.start()] if a_m else s[eor:]
    orphans = _re.findall(r'"([^"]*)"', middle)
    if orphans: reasoning = f"{reasoning} {' '.join(orphans)}".strip()
    if not a_m: return None
    ans = a_m.group(1) or a_m.group(2) or a_m.group(3)
    conf = float(c_m.group(1)) if c_m else 50
    return {'reasoning': reasoning, 'answer': ans, 'confidence': conf}

def _fb_regex(raw):
    r_m = _re.search(r'"reasoning"\s*:\s*"((?:[^"\\]|\\.)*?)"', raw, _re.DOTALL)
    a_m = _re.search(r'"answer"\s*:\s*(?:"((?:[^"\\]|\\.)*?)"|([-+]?\d+(?:\.\d+)?))', raw, _re.DOTALL)
    c_m = _re.search(r'"confidence"\s*:\s*(\d+(?:\.\d+)?)', raw)
    if r_m is None or a_m is None: return None
    return {'reasoning': r_m.group(1), 'answer': a_m.group(1) or a_m.group(2),
            'confidence': float(c_m.group(1)) if c_m else 50}

def extract_json_from_response(raw):
    if not raw or not isinstance(raw, str): return None
    try:
        parsed = _original_extract_json(raw)
        if parsed and isinstance(parsed, dict) and 'reasoning' in parsed and 'answer' in parsed:
            return parsed
    except Exception: pass
    for strategy in (_fb_markdown, _fb_literal_newline, _fb_concatenated, _fb_regex):
        try:
            parsed = strategy(raw)
            if parsed and isinstance(parsed, dict) and 'reasoning' in parsed and 'answer' in parsed:
                return parsed
        except Exception: continue
    return None


# ---------- 7.4 — Model yükleme ve temizleme ----------
def load_model_4bit(model_id: str):
    print(f"► Model yükleniyor: {model_id}")
    print("  (Llama 3.1 8B: ilk indirme ~5-10 dk, sonraki çalıştırmalar cache'den hızlı)")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print(f"✓ Model yüklendi: {model_id}")
    return model, tokenizer

def cleanup_model_and_gpu():
    """LLM'i GPU'dan tamamen at, generation sonrası skorlama için yer aç."""
    for var_name in ['model', 'tokenizer']:
        if var_name in globals():
            globals()[var_name] = None
            try: del globals()[var_name]
            except: pass
    for _ in range(3):
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    free = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"✓ GPU temizlendi. Boş: {free:.2f} GB")

print("✓ Yardımcı fonksiyonlar tanımlandı")
print("  - build_user_prompt, generate_response")
print("  - extract_json_from_response (4 fallback'li)")
print("  - load_model_4bit, cleanup_model_and_gpu")

✓ Yardımcı fonksiyonlar tanımlandı
  - build_user_prompt, generate_response
  - extract_json_from_response (4 fallback'li)
  - load_model_4bit, cleanup_model_and_gpu


---
## 🦙 8. Llama'yı yükle

Beklenen süre: İlk seferde ~5-10 dk (model indirme + 4-bit quantize). Sonraki çalıştırmalarda cache'den ~1 dk.
---

In [ ]:
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_SHORT_NAME = "llama31_8b"
MODEL_DISPLAY_NAME = "Llama 3.1 8B Instruct (varyant)"

model, tokenizer = load_model_4bit(MODEL_ID)

# Yükleme sonrası bellek durumu
used = torch.cuda.memory_allocated() / 1024**3
free = torch.cuda.mem_get_info()[0] / 1024**3
print(f"\n✓ GPU kullanım: {used:.2f} GB | Boş: {free:.2f} GB")

► Model yükleniyor: meta-llama/Llama-3.1-8B-Instruct
  (Llama 3.1 8B: ilk indirme ~5-10 dk, sonraki çalıştırmalar cache'den hızlı)


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

✓ Model yüklendi: meta-llama/Llama-3.1-8B-Instruct

✓ GPU kullanım: 5.31 GB | Boş: 16.42 GB


---
## 🏃 9. Generation çalıştır (checkpoint'li, ~1.5 saat)

Her 5 soruda Drive'a kaydeder. Kesilirse hücreyi tekrar çalıştır, kaldığı yerden devam eder.
---

In [ ]:
from tqdm.notebook import tqdm

checkpoint_file = f"{CHECKPOINT_PATH}generation_{MODEL_SHORT_NAME}_{DATASET_TAG}_{TODAY}.json"

# Daha önce işlenmiş soruları yükle (varsa)
results = []
processed_ids = set()
if os.path.exists(checkpoint_file):
    try:
        with open(checkpoint_file, 'r', encoding='utf-8') as f:
            results = json.load(f)
        processed_ids = {r['question_id'] for r in results}
        print(f"✓ Checkpoint bulundu: {len(processed_ids)} soru işlenmiş, devam ediliyor")
    except json.JSONDecodeError:
        print("⚠ Checkpoint bozuk, baştan başlanıyor")
        results = []
        processed_ids = set()

to_process = df[~df['question_id'].isin(processed_ids)]

if len(to_process) == 0:
    print("✓ Tüm sorular işlenmiş, generation aşaması atlanıyor")
else:
    print(f"► {len(to_process)} soru işlenecek")
    for idx, row in tqdm(to_process.iterrows(), total=len(to_process), desc=f"Gen [{MODEL_SHORT_NAME}]"):
        context = str(row['context'])
        question = str(row['question'])
        try:
            raw = generate_response(model, tokenizer, context, question)
            parsed = extract_json_from_response(raw)
            if parsed is None or not isinstance(parsed, dict):
                results.append({
                    'question_id': row['question_id'],
                    'model_raw_response': raw,
                    'model_reasoning': None,
                    'model_answer': None,
                    'model_confidence': None,
                    'error': 'JSON_PARSE_ERROR',
                })
            else:
                results.append({
                    'question_id': row['question_id'],
                    'model_raw_response': raw,
                    'model_reasoning': parsed.get('reasoning'),
                    'model_answer': parsed.get('answer'),
                    'model_confidence': parsed.get('confidence'),
                    'error': None,
                })
        except Exception as e:
            results.append({
                'question_id': row['question_id'],
                'model_raw_response': '',
                'model_reasoning': None,
                'model_answer': None,
                'model_confidence': None,
                'error': str(e),
            })
        # Her 5 soruda kaydet
        if len(results) % 5 == 0:
            with open(checkpoint_file, 'w', encoding='utf-8') as f:
                json.dump(results, f, ensure_ascii=False, indent=2)

    # Son kayıt
    with open(checkpoint_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"\n✓ Checkpoint kaydedildi: {checkpoint_file}")

# Özet
results_df = pd.DataFrame(results)
merged = df.merge(results_df, on='question_id', how='left')
success_count = merged['model_answer'].notna().sum()
error_count = merged['error'].notna().sum()
print(f"\n✓ Generation özeti:")
print(f"   Toplam: {len(merged)}")
print(f"   Başarılı: {success_count}")
print(f"   Hatalı: {error_count}")

► 257 soru işlenecek


Gen [llama31_8b]:   0%|          | 0/257 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(



✓ Checkpoint kaydedildi: /content/drive/MyDrive/BitirmeProjesi/checkpoints/generation_llama31_8b_varyant_20260517.json

✓ Generation özeti:
   Toplam: 257
   Başarılı: 239
   Hatalı: 18


---
## 🧹 10. GPU'yu temizle (Llama'yı bellekten at)

Skorlama CPU'da yapılacak, GPU'ya artık ihtiyaç yok. Ama Llama'yı bellekten silmek hijyen meselesi.
---

In [ ]:
cleanup_model_and_gpu()

✓ GPU temizlendi. Boş: 21.69 GB


---
## 📐 11. Checkpoint'li skorlama (her 10 soruda kayıt)

Qwen'de yaşadığımız "kesilince baştan başlama" sorunu burada yok. Internet keserse veya runtime durursa, bu hücreyi tekrar çalıştır, kaldığı yerden devam eder.
---

In [ ]:
import numpy as np

# numpy → Python native tip dönüşümü (JSON serializable için)
def to_jsonable(obj):
    if isinstance(obj, (np.floating, np.float32, np.float64)):
        return float(obj)
    if isinstance(obj, (np.integer, np.int32, np.int64)):
        return int(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, dict):
        return {k: to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(x) for x in obj]
    return obj


def score_one_row(row):
    """Tek bir satır için tüm metrikleri hesapla (faithfulness + accuracy + epistemic + soft acc)."""
    result = {'question_id': row['question_id']}
    try:
        evidence_list = row['gold_evidence_list']
        reasoning = str(row['model_reasoning'])

        # Faithfulness
        faith = calculate_faithfulness(reasoning, evidence_list)
        result['semantic_similarity'] = float(faith['semantic_similarity'])
        result['nli_score'] = float(faith['nli_score'])
        result['faithfulness_score'] = float(faith['faithfulness_score'])
        result['evidence_recall'] = float(faith['evidence_recall'])

        # Exact accuracy
        m_ans = str(row['model_answer']).strip().lower()
        g_ans = str(row['gold_answer']).strip().lower()
        result['accuracy_score'] = 1 if m_ans == g_ans else 0

        # Confidence normalize
        try:
            conf_raw = float(row['model_confidence']) if row['model_confidence'] is not None else 50
        except (ValueError, TypeError):
            conf_raw = 50
        result['confidence_normalized'] = conf_raw / 100

        # Epistemic metrics
        ep = calculate_epistemic_metrics(
            conf_raw,
            result['faithfulness_score'],
            result['accuracy_score']
        )
        for k, v in ep.items():
            result[k] = to_jsonable(v)

        # Soft accuracy
        soft = calculate_soft_accuracy(row['model_answer'], row['gold_answer'])
        result['soft_match'] = bool(soft['soft_match'])
        result['soft_accuracy_score'] = int(soft['accuracy_score'])
        result['match_type'] = str(soft['match_type'])
        result['answer_semantic_sim'] = float(soft['semantic_sim'])
        result['answer_nli_entailment'] = float(soft['nli_entailment'])

        result['error'] = None
    except Exception as e:
        result['error'] = str(e)
    return result


# Generation sonuçlarını oku
with open(checkpoint_file, 'r', encoding='utf-8') as f:
    gen_results = json.load(f)
gen_df = pd.DataFrame(gen_results)
merged = df.merge(gen_df, on='question_id', how='left')

# Geçerli satırları al
valid = merged[
    merged['error'].isna() &
    merged['model_reasoning'].notna() &
    (merged['model_reasoning'].astype(str).str.len() > 0)
].copy().reset_index(drop=True)
print(f"► Skorlanacak: {len(valid)} / {len(merged)}")

# Skorlama checkpoint
scoring_checkpoint = f"{CHECKPOINT_PATH}scoring_{MODEL_SHORT_NAME}_{DATASET_TAG}_{TODAY}.json"
scored_dict = {}
if os.path.exists(scoring_checkpoint):
    try:
        with open(scoring_checkpoint, 'r', encoding='utf-8') as f:
            scored_list = json.load(f)
        scored_dict = {r['question_id']: r for r in scored_list}
        print(f"✓ Skorlama checkpoint bulundu: {len(scored_dict)} zaten skorlanmış")
    except json.JSONDecodeError:
        print("⚠ Skorlama checkpoint bozuk, baştan başlanıyor")
        scored_dict = {}

to_score = valid[~valid['question_id'].isin(scored_dict.keys())]
print(f"► İşlenecek: {len(to_score)} soru\n")

# Skorla
for _, row in tqdm(to_score.iterrows(), total=len(to_score), desc="Skorlama"):
    res = score_one_row(row)
    scored_dict[res['question_id']] = res
    if len(scored_dict) % 10 == 0:
        with open(scoring_checkpoint, 'w', encoding='utf-8') as f:
            json.dump(list(scored_dict.values()), f, ensure_ascii=False, indent=2)

# Son kayıt
with open(scoring_checkpoint, 'w', encoding='utf-8') as f:
    json.dump(list(scored_dict.values()), f, ensure_ascii=False, indent=2)
print(f"\n✓ Skorlama checkpoint kaydedildi: {scoring_checkpoint}")

► Skorlanacak: 239 / 257
✓ Skorlama checkpoint bulundu: 10 zaten skorlanmış
► İşlenecek: 229 soru



Skorlama:   0%|          | 0/229 [00:00<?, ?it/s]


✓ Skorlama checkpoint kaydedildi: /content/drive/MyDrive/BitirmeProjesi/checkpoints/scoring_llama31_8b_varyant_20260517.json


---
## 📊 12. Final CSV + Özet
---

In [ ]:
# CSV oluştur
scored_df = pd.DataFrame(list(scored_dict.values()))
final_df = valid.merge(scored_df, on='question_id', how='left')

output_path = f"{RESULTS_PATH}evaluation_acik_{MODEL_SHORT_NAME}_{DATASET_TAG}_{TODAY}.csv"
final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"✓ CSV kaydedildi: {output_path}")

# Özet
print("\n" + "=" * 70)
print(f"📊 ÖZET — {MODEL_DISPLAY_NAME}")
print("=" * 70)
print(f"Toplam soru: {len(final_df)}")

if 'accuracy_score' in final_df.columns:
    exact = final_df['accuracy_score'].mean() * 100
    print(f"\nExact Accuracy: {exact:.1f}%  ({int(final_df['accuracy_score'].sum())}/{len(final_df)})")

if 'soft_accuracy_score' in final_df.columns:
    soft = final_df['soft_accuracy_score'].mean() * 100
    print(f"Soft Accuracy:  {soft:.1f}%  ({int(final_df['soft_accuracy_score'].sum())}/{len(final_df)})")

if 'faithfulness_score' in final_df.columns:
    print(f"\nOrtalama semantic_similarity: {final_df['semantic_similarity'].mean():.3f}")
    print(f"Ortalama nli_score:           {final_df['nli_score'].mean():.3f}")
    print(f"Ortalama faithfulness_score:  {final_df['faithfulness_score'].mean():.3f}")
    print(f"Ortalama evidence_recall:     {final_df['evidence_recall'].mean():.3f}")

if 'confidence_normalized' in final_df.columns:
    print(f"\nOrtalama confidence: {final_df['confidence_normalized'].mean()*100:.1f}%")
    print(f"Ortalama epistemic_gap: {final_df['epistemic_gap'].mean():.3f}")

    print("\nHallucination risk dağılımı:")
    for risk, cnt in final_df['hallucination_risk'].value_counts().items():
        print(f"   {risk}: {cnt} ({cnt/len(final_df)*100:.1f}%)")

    print("\nCalibration tipi dağılımı:")
    for cal, cnt in final_df['calibration_type'].value_counts().items():
        print(f"   {cal}: {cnt} ({cnt/len(final_df)*100:.1f}%)")

print("=" * 70)

✓ CSV kaydedildi: /content/drive/MyDrive/BitirmeProjesi/results/evaluation_acik_llama31_8b_varyant_20260517.csv

📊 ÖZET — Llama 3.1 8B Instruct (varyant)
Toplam soru: 239

Exact Accuracy: 20.5%  (49/239)
Soft Accuracy:  39.3%  (72/239)

Ortalama semantic_similarity: 0.869
Ortalama nli_score:           0.505
Ortalama faithfulness_score:  0.651
Ortalama evidence_recall:     1.000

Ortalama confidence: 97.9%
Ortalama epistemic_gap: 0.329

Hallucination risk dağılımı:
   ORTA: 157 (65.7%)
   DÜŞÜK: 67 (28.0%)
   YÜKSEK: 15 (6.3%)

Calibration tipi dağılımı:
   Overconfident: 224 (93.7%)
   İyi Kalibre: 15 (6.3%)


---
## 🔄 13. (Opsiyonel) Sadece skorlamayı yeniden yap

Eğer generation bitti ve sadece skorlama hücresi kesildiyse, bu hücreyi çalıştır. Llama'yı tekrar yüklemez, sadece skorlama yapar.

Önkoşul: 3, 4, 5, 6, 7. hücreleri çalıştırmış olman gerekir.
---

In [ ]:
# Bu hücre sadece şu durumda gerekli:
# - Generation tamamlandı (9. hücre)
# - Skorlama hücresi (11) kesildi veya kapandı
# - Tekrar başlattığında 3, 4, 5, 6, 7'yi çalıştırdın
# - 8, 9, 10'a gerek yok (Llama yüklemeye gerek yok)
# Sonra bu hücreyi çalıştır → 11 ve 12'deki mantığı tekrar uygular

# Generation checkpoint'ini bul
import glob
candidates = glob.glob(f"{CHECKPOINT_PATH}generation_{MODEL_SHORT_NAME}_{DATASET_TAG}_*.json")
if not candidates:
    raise FileNotFoundError(f"Generation checkpoint bulunamadı: {MODEL_SHORT_NAME}")
checkpoint_file = sorted(candidates)[-1]
print(f"✓ Generation checkpoint: {checkpoint_file}")

with open(checkpoint_file, 'r', encoding='utf-8') as f:
    gen_results = json.load(f)
gen_df = pd.DataFrame(gen_results)
merged = df.merge(gen_df, on='question_id', how='left')

valid = merged[
    merged['error'].isna() &
    merged['model_reasoning'].notna() &
    (merged['model_reasoning'].astype(str).str.len() > 0)
].copy().reset_index(drop=True)
print(f"► Skorlanacak: {len(valid)} / {len(merged)}")

scoring_checkpoint = f"{CHECKPOINT_PATH}scoring_{MODEL_SHORT_NAME}_{DATASET_TAG}_{TODAY}.json"
scored_dict = {}
if os.path.exists(scoring_checkpoint):
    try:
        with open(scoring_checkpoint, 'r', encoding='utf-8') as f:
            scored_list = json.load(f)
        scored_dict = {r['question_id']: r for r in scored_list}
        print(f"✓ Önceki skorlama: {len(scored_dict)} zaten yapılmış")
    except json.JSONDecodeError:
        scored_dict = {}

to_score = valid[~valid['question_id'].isin(scored_dict.keys())]
print(f"► İşlenecek: {len(to_score)} soru\n")

for _, row in tqdm(to_score.iterrows(), total=len(to_score), desc="Re-Skorlama"):
    res = score_one_row(row)
    scored_dict[res['question_id']] = res
    if len(scored_dict) % 10 == 0:
        with open(scoring_checkpoint, 'w', encoding='utf-8') as f:
            json.dump(list(scored_dict.values()), f, ensure_ascii=False, indent=2)

with open(scoring_checkpoint, 'w', encoding='utf-8') as f:
    json.dump(list(scored_dict.values()), f, ensure_ascii=False, indent=2)

# CSV + özet (12. hücredekiyle aynı)
scored_df = pd.DataFrame(list(scored_dict.values()))
final_df = valid.merge(scored_df, on='question_id', how='left')
output_path = f"{RESULTS_PATH}evaluation_acik_{MODEL_SHORT_NAME}_{DATASET_TAG}_{TODAY}.csv"
final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"\n✓ CSV: {output_path}")
print(f"  Exact: {final_df['accuracy_score'].mean()*100:.1f}% | Soft: {final_df['soft_accuracy_score'].mean()*100:.1f}%")
print(f"  Faithfulness: {final_df['faithfulness_score'].mean():.3f} | Confidence: {final_df['confidence_normalized'].mean()*100:.1f}%")